<a href="https://colab.research.google.com/github/simjonghyeon04/-/blob/main/%EC%99%80%EC%9D%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 필수 라이브러리 설치 (Gradio가 설치되어 있지 않은 경우를 대비)
!pip install -q gradio scikit-learn pandas

import gradio as gr
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# 2. 데이터 로드 및 전처리 (Feature Selection 3개)
# ==========================================
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)

# 요구사항: Feature Selection 3개만 선택
# 와인 데이터셋에서 가장 직관적이고 영향력이 큰 3가지 특성을 선택했습니다.
# (알코올 도수, 말산, 마그네슘)
selected_features = ['alcohol', 'malic_acid', 'magnesium']
X = df[selected_features]
y = wine.target

# ==========================================
# 3. 데이터 훈련 및 예측 모델 생성
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 생성 및 학습
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# 와인 클래스 이름 매핑 (Class 0, Class 1, Class 2 -> 실제 타겟 이름)
target_names = wine.target_names

# ==========================================
# 4. Gradio 예측 함수 정의
# ==========================================
def predict_wine(alcohol, malic_acid, magnesium):
    # 입력 데이터를 데이터프레임 형태로 변환
    input_data = pd.DataFrame([[alcohol, malic_acid, magnesium]], columns=selected_features)

    # 예측 수행
    prediction = model.predict(input_data)[0]
    proba = model.predict_proba(input_data)[0]

    # 결과 반환 (가장 높은 확률의 와인 클래스 이름과 전체 확률)
    result_text = f"이 와인은 **'{target_names[prediction].upper()}'** 일 확률이 가장 높습니다."
    prob_dict = {target_names[i]: float(proba[i]) for i in range(len(target_names))}

    return result_text, prob_dict

# ==========================================
# 5. Gradio 인터페이스 구성 및 마운팅 (Single Server)
# ==========================================
# 각 feature의 대략적인 데이터 범위를 슬라이더의 min, max로 지정했습니다.
interface = gr.Interface(
    fn=predict_wine,
    inputs=[
        gr.Slider(minimum=11.0, maximum=15.0, value=13.0, step=0.1, label="Alcohol (알코올 도수)"),
        gr.Slider(minimum=0.5, maximum=6.0, value=2.3, step=0.1, label="Malic Acid (말산 함량)"),
        gr.Slider(minimum=70.0, maximum=160.0, value=100.0, step=1.0, label="Magnesium (마그네슘 함량)")
    ],
    outputs=[
        gr.Markdown(label="예측 결과"),
        gr.Label(label="클래스별 확률")
    ],
    title="🍷 Wine Type Predictor (와인 종류 예측 서비스)",
    description="3가지 주요 특성(Alcohol, Malic Acid, Magnesium)을 입력하여 와인의 종류(class_0, class_1, class_2)를 예측합니다."
)

# Colab 환경에서 외부 링크(public URL) 생성을 위해 share=True 설정
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a745ebf1833ba5bd58.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
